In [0]:
import sys
import importlib

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import universe
import feature_engineer

importlib.reload(config)
importlib.reload(universe)
importlib.reload(feature_engineer)
import news_feature_builder
importlib.reload(news_feature_builder)

from config import RAW_TABLE_NAME, FEATURE_TABLE_NAME, RAW_NEWS_TABLE_NAME, USE_NEWS_FEATURES
from news_feature_builder import NewsFeatureBuilder
from universe import TRAIN_UNIVERSE, MARKET_CONTEXT
from feature_engineer import FeatureEngineer
from news_feature_builder import NewsFeatureBuilder

In [0]:
raw_df = spark.table(RAW_TABLE_NAME).toPandas()

print("Raw rows:", len(raw_df))
print("Raw cols:", len(raw_df.columns))
raw_df.head()

In [0]:
engineer = FeatureEngineer(
    universe=TRAIN_UNIVERSE,
    market_context=MARKET_CONTEXT
)

features_df = engineer.build_stock_specific_feature_store(raw_df)
engineer.validate_feature_store(features_df)

print("Feature store rows:", len(features_df))
print("Feature store cols:", len(features_df.columns))
features_df.head()

In [0]:
if USE_NEWS_FEATURES:
    try:
        news_raw_df = spark.table(RAW_NEWS_TABLE_NAME).toPandas()

        news_builder = NewsFeatureBuilder(train_universe=TRAIN_UNIVERSE)
        daily_news_features_df = news_builder.build_daily_news_features(news_raw_df)

        features_df = engineer.merge_news_features(features_df, daily_news_features_df)

        print("News features merged.")
        print("Daily news feature columns:")
        print([c for c in daily_news_features_df.columns if c not in ["Date", "Ticker"]][:20])

    except Exception as e:
        print("Could not merge news features. Continuing without news.")
        print("Reason:", e)

In [0]:
print(features_df.columns.tolist())

In [0]:
null_summary = features_df.isnull().sum().sort_values(ascending=False)
null_summary.head(20)

In [0]:
spark.createDataFrame(features_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(FEATURE_TABLE_NAME)

In [0]:
spark.sql(f"SELECT * FROM {FEATURE_TABLE_NAME} LIMIT 3").show()

In [0]:
print("Feature store rows:", len(features_df))
print("Feature store cols:", len(features_df.columns))

In [0]:
print([c for c in features_df.columns if "news" in c.lower() or "sentiment" in c.lower()][:50])

In [0]:
features_df[features_df["Ticker"] == "AAPL"][[
    "Date",
    "news_count_1d",
    "news_count_3d",
    "news_count_5d",
    "headline_sentiment_mean_1d",
    "combined_sentiment_mean_1d",
    "abnormal_news_count_20d"
]].tail(10)